# Multi-Target NILM on IAWE Dataset
**Goal**: Disaggregate Washing Machine and Air Conditioner from the Mains using a single 1D Convolutional Neural Network (Seq2Point architecture).
**Steps**:
1. Data Loading (Custom HDF5 Reader)
2. Exploratory Data Analysis (EDA)
3. Preprocessing (Resampling, Imputation, Windowing)
4. Deep Learning Model (PyTorch)
5. Evaluation


In [ ]:
import tables
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## 1. Data Loading
The IAWE dataset has HDF5 string encoding issues with newer versions of pandas. We use a custom reader using `tables` to bypass this.

In [ ]:
def load_meter_data(h5_path, meter_key):
    '''Loads index and active power from the HDF5 file using tables.'''
    with tables.open_file(h5_path, mode='r') as f:
        node = f.get_node(meter_key + '/table')
        
        # Read the entire array into memory (IAWE meters aren't too huge individually)
        data = node[:]
        
        # 'index' is Unix timestamp in nanoseconds
        index = data['index']
        # 'values_block_0' contains 7 measurements, Active Power is the first one (index 0)
        # as verified from the node metadata attributes
        active_power = data['values_block_0'][:, 0]
        
    # Convert index to datetime
    dates = pd.to_datetime(index, unit='ns').tz_localize('UTC').tz_convert('Asia/Kolkata')
    
    # Create Series
    series = pd.Series(data=active_power, index=dates, name='power')
    # Filter out negative anomalies
    series[series < 0] = 0 
    return series

h5_path = '/data/Projects/NILM/IAWE/Data/iawe.h5'

print("Loading Mains (Meter 1)...")
mains = load_meter_data(h5_path, '/building1/elec/meter1')

print("Loading Washing Machine (Meter 6)...")
washing_machine = load_meter_data(h5_path, '/building1/elec/meter6')

print("Loading Air Conditioner (Meter 4)...")
ac = load_meter_data(h5_path, '/building1/elec/meter4')


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Let's align them into a single dataframe first using 1-minute resampling to view clearly.
# For EDA, 1-min resample is fast and sufficient.
print("Resampling for EDA...")
df_eda = pd.DataFrame({
    'Mains': mains.resample('1min').mean().ffill(),
    'Washing_Machine': washing_machine.resample('1min').mean().ffill(),
    'AC': ac.resample('1min').mean().ffill()
}).dropna()

print(f"EDA Data shape: {df_eda.shape}")
df_eda.head()


In [ ]:
# 1. Plotting a small window of consumption
start_date = df_eda.index[0] + pd.Timedelta(days=5)
end_date = start_date + pd.Timedelta(days=2)

subset = df_eda.loc[start_date:end_date]

plt.figure(figsize=(15, 6))
plt.plot(subset.index, subset['Mains'], label='Mains', alpha=0.7)
plt.plot(subset.index, subset['Washing_Machine'], label='Washing Machine', alpha=0.9)
plt.plot(subset.index, subset['AC'], label='Air Conditioner', alpha=0.9)
plt.title("Power Consumption over 2 Days")
plt.xlabel("Time")
plt.ylabel("Power (W)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 2. Daily Energy Consumption Boxplots (kWh)
# Multiply by (1/60) to convert Watts per minute to Wh, then divide by 1000 for kWh
daily_energy = df_eda.resample('D').sum() * (1/60) / 1000

plt.figure(figsize=(10, 5))
sns.boxplot(data=daily_energy)
plt.title("Daily Energy Consumption Distribution (kWh)")
plt.ylabel("Energy (kWh)")
plt.tight_layout()
plt.show()


## 3. Preprocessing
We will create sequences of length `W` from the `Mains` power. The targets will be the `Washing_Machine` and `AC` power at the center of the window (Seq2Point architecture).

In [ ]:
WINDOW_SIZE = 99

# We'll use df_eda which is already 1-minute resampled and ffilled.
data = df_eda.copy()

# Standardize data
scaler_mains = StandardScaler()
scaler_wm = StandardScaler()
scaler_ac = StandardScaler()

data['Mains_norm'] = scaler_mains.fit_transform(data[['Mains']])
data['WM_norm'] = scaler_wm.fit_transform(data[['Washing_Machine']])
data['AC_norm'] = scaler_ac.fit_transform(data[['AC']])

# We only need a slice for training to keep execution fast. Let's use 40 days for train, 10 for test.
train_size = int(len(data) * 0.7)
train_data = data.iloc[:train_size]
test_data = data.iloc[train_size:]

print(f"Train size: {len(train_data)} | Test size: {len(test_data)}")


In [ ]:
def create_sequences(df, window_size):
    '''Creates sliding windows for X and targets for Y'''
    X, Y = [], []
    mains_vals = df['Mains_norm'].values
    wm_vals = df['WM_norm'].values
    ac_vals = df['AC_norm'].values
    
    pad = window_size // 2
    
    for i in range(len(mains_vals) - window_size):
        X.append(mains_vals[i : i + window_size])
        # Target is the center point
        Y.append([wm_vals[i + pad], ac_vals[i + pad]])
        
    return np.array(X), np.array(Y)

print("Creating training sequences...")
X_train, Y_train = create_sequences(train_data, WINDOW_SIZE)
print("Creating testing sequences...")
X_test, Y_test = create_sequences(test_data, WINDOW_SIZE)

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")


In [ ]:
class NILMDataset(Dataset):
    def __init__(self, X, Y):
        # Shape: (Batch, Channels, Length) -> (Batch, 1, 99)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.Y = torch.tensor(Y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

batch_size = 512
train_loader = DataLoader(NILMDataset(X_train, Y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(NILMDataset(X_test, Y_test), batch_size=batch_size, shuffle=False)


## 4. Deep Learning Model
Multi-Target Sequence-to-Point CNN.
Input: (Batch, 1, 99)
Output: (Batch, 2) [Washing Machine, AC]

In [ ]:
class MultiTargetSeq2Point(nn.Module):
    def __init__(self, window_size):
        super(MultiTargetSeq2Point, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=30, kernel_size=10, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv1d(in_channels=30, out_channels=30, kernel_size=8, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv1d(in_channels=30, out_channels=40, kernel_size=6, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv1d(in_channels=40, out_channels=50, kernel_size=5, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv1d(in_channels=50, out_channels=50, kernel_size=5, stride=1, padding=0),
            nn.ReLU()
        )
        
        # Calculate flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, 1, window_size)
            flat_size = self.conv(dummy).view(1, -1).shape[1]
            
        self.fc = nn.Sequential(
            nn.Linear(flat_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2)  # 2 outputs: Washing Machine and AC
        )
        
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = MultiTargetSeq2Point(WINDOW_SIZE).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print(model)


## 5. Training Loop

In [ ]:
epochs = 5 # Small number for demonstration, increase for better accuracy

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        
    train_loss /= len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss (MSE): {train_loss:.4f}")


## 5.5 Save the Model

In [ ]:
# Save the trained model's state dictionary
model_save_path = 'nilm_multi_target_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model successfully saved to {model_save_path}")


## 6. Evaluation (Using Saved Model)

In [ ]:
# Load the model from disk for evaluation
loaded_model = MultiTargetSeq2Point(WINDOW_SIZE).to(device)
loaded_model.load_state_dict(torch.load(model_save_path))
loaded_model.eval()

predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs = loaded_model(inputs).cpu().numpy()
        predictions.append(outputs)
        actuals.append(targets.numpy())

predictions = np.concatenate(predictions, axis=0)
actuals = np.concatenate(actuals, axis=0)

# Un-normalize
pred_wm = scaler_wm.inverse_transform(predictions[:, 0].reshape(-1, 1)).flatten()
pred_ac = scaler_ac.inverse_transform(predictions[:, 1].reshape(-1, 1)).flatten()

actual_wm = scaler_wm.inverse_transform(actuals[:, 0].reshape(-1, 1)).flatten()
actual_ac = scaler_ac.inverse_transform(actuals[:, 1].reshape(-1, 1)).flatten()

# Ensure no negative power predictions
pred_wm[pred_wm < 0] = 0
pred_ac[pred_ac < 0] = 0

print("MAE - Washing Machine:", mean_absolute_error(actual_wm, pred_wm))
print("MAE - AC:             ", mean_absolute_error(actual_ac, pred_ac))

# Signal Aggregate Error (SAE)
sae_wm = abs(pred_wm.sum() - actual_wm.sum()) / actual_wm.sum()
sae_ac = abs(pred_ac.sum() - actual_ac.sum()) / actual_ac.sum()

print("SAE - Washing Machine:", sae_wm)
print("SAE - AC:             ", sae_ac)


In [ ]:
# Plot some results
start_idx = 1000
end_idx = 2000

time_axis = test_data.index[WINDOW_SIZE//2 : -WINDOW_SIZE//2][start_idx:end_idx]

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Washing Machine Plot
axes[0].plot(time_axis, actual_wm[start_idx:end_idx], label='Ground Truth', alpha=0.7)
axes[0].plot(time_axis, pred_wm[start_idx:end_idx], label='Prediction', alpha=0.7)
axes[0].set_title('Washing Machine Disaggregation')
axes[0].set_ylabel('Power (W)')
axes[0].legend()

# AC Plot
axes[1].plot(time_axis, actual_ac[start_idx:end_idx], label='Ground Truth', alpha=0.7)
axes[1].plot(time_axis, pred_ac[start_idx:end_idx], label='Prediction', alpha=0.7)
axes[1].set_title('Air Conditioner Disaggregation')
axes[1].set_ylabel('Power (W)')
axes[1].set_xlabel('Time')
axes[1].legend()

plt.tight_layout()
plt.show()
